<div style="text-align:center;">
  <b>Национальный исследовательский университет ИТМО</b><br>
  Факультет информационных технологий и программирования<br>
  Кафедра Компьютерных Технологий
</div>


---

# **Методы оптимизации**  
### Лабораторная работа №3 
### Метод стохастического градиентного спуска (SGD) и его модификации

<div style="text-align:right; font-size:12px">
Выполнили:<br>
Алфёров Кирилл М3232<br>
Салов Егор М3232
</div>





### Информация о датасете

Датасет содержит 9358 записей с почасовыми усреднёнными откликами от массива из 5 химических сенсоров на основе металлических оксидов, встроенных в устройство химического мониторинга качества воздуха. Устройство было размещено на улице в зоне с высоким уровнем загрязнения, на уровне дороги, в одном из городов Италии. Данные собирались с марта 2004 по февраль 2005 года (в течение одного года), что делает этот набор данных одним из самых продолжительных по длительности свободно доступных полевых наблюдений с использованием химических сенсоров.

Эталонные (ground truth) данные о почасовых усреднённых концентрациях угарного газа (CO), неметановых углеводородов, бензола, суммарных оксидов азота (NOx) и диоксида азота (NO₂) были получены с помощью сертифицированного анализатора, установленного рядом.

В данных присутствуют признаки перекрёстной чувствительности сенсоров, а также концептуального и сенсорного дрейфа, что описано в статье De Vito и др., Sens. and Act. B, Vol. 129, №2, 2008 (необходима ссылка на источник). Эти эффекты могут снижать точность оценки концентраций загрязняющих веществ по данным сенсоров. Пропущенные значения помечены как -200.

### Читаем датасет

In [ ]:
import pandas

dataset = pandas.read_csv("dataset/AirQualityUCI.csv", sep=';', decimal=',')
dataset

### Вычислим корреляцию параметров попарно

In [ ]:
dataset.corr(numeric_only=True)

### Что берём?


Для нашей регрессионной модели мы решили предсказывать именно **C6H6(GT)** (концентрацию бензола, мкг/м³). Вот почему:

1. **Очень сильная линейная связь**  
   - С абсолютной влажностью (AH): *r* ≈ 0.985  
   - С температурой воздуха (T): *r* ≈ 0.971  
   - С относительной влажностью (RH): *r* ≈ 0.925  
   
   Когда |*r*| > 0.9, простая линейная модель уже способна выдавать точные и стабильные предсказания.

2. **Минимум избыточности среди признаков**  
   Мы исключили сенсоры, которые почти полностью дублировали друг друга (корреляция > 0.9 между ними). В результате в качестве входных факторов остались только самые информативные переменные без сильной взаимозависимости


Итак, зависимая переменная **y = C6H6(GT)**, независимые переменные **x = AH (|r| = 0.985), PT08.S1(CO) (|r| = 0.853), PT08.S3(NOx) (|r| = 0.512)** (а также у датчиков PT08.S1(CO) и PT08.S3(NOx) слабая корреляция |r| = 0.087, что снижает мультиколлинеарность)




## 3 пункт

In [ ]:
import torch
import pyTorch.Torch as Torch
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

# Загрузка и подготовка данных
X, y = load_diabetes(return_X_y=True)

# Масштабирование (важно для стабильного градиентного спуска)
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).ravel()

# Делим на train/test
x_tr, x_te, y_tr, y_te = train_test_split(X_scaled, y_scaled, test_size=0.2, random_state=42)

# Перевод в torch.FloatTensor
xtr_t = torch.FloatTensor(x_tr)
xte_t = torch.FloatTensor(x_te)
ytr_t = torch.FloatTensor(y_tr).reshape(-1, 1)
yte_t = torch.FloatTensor(y_te).reshape(-1, 1)

learning_rates = {
    "SGD": 1e-2,
    "SGD+Momentum": 1e-2,
    "SGD+Nesterov": 1e-2,
    "Adagrad": 1e-2,
    "RMSprop": 1e-2,
    "Adam": 1e-2,
}

optimisers = list(learning_rates.keys())

results = []
for name in optimisers:
    res = Torch.train_torch(
        opt_name=name,
        xtr=xtr_t,
        ytr=ytr_t,
        xte=xte_t,
        yte=yte_t,
        epochs=50,
        batch_size=32,
        lr_dict=learning_rates
    )
    results.append(res)
    print(f"{name: <14} | MSE: {res['mse']:.4f} | Time: {res['time']:.2f}s | Mem: {res['memory']:.1f}MB")


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

# Ваши результаты в виде словаря
results = [
    {"optimizer": "SGD", "mse": 0.4973, "time": 0.65},
    {"optimizer": "SGD+Momentum", "mse": 0.6735, "time": 0.48},
    {"optimizer": "SGD+Nesterov", "mse": 0.5176, "time": 0.46},
    {"optimizer": "Adagrad", "mse": 0.5436, "time": 0.87},
    {"optimizer": "RMSprop", "mse": 0.5236, "time": 1.36},
    {"optimizer": "Adam", "mse": 0.4821, "time": 1.43},
]

df = pd.DataFrame(results)

# График: MSE по оптимизатору
plt.figure(figsize=(10, 5))
plt.bar(df["optimizer"], df["mse"])
plt.ylabel("MSE (Mean Squared Error)")
plt.title("Сравнение оптимизаторов по MSE")
plt.grid(axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# График: Время по оптимизатору
plt.figure(figsize=(10, 5))
plt.bar(df["optimizer"], df["time"])
plt.ylabel("Время обучения (сек)")
plt.title("Сравнение оптимизаторов по времени обучения")
plt.grid(axis="y")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## Дополнительное задание №2
## Метод опорных векторов(SVM)

Была реализована модель бинарной классификации на основе **метода опорных векторов (Support Vector Machine, SVM)** с **линейным ядром**. Цель эксперимента — визуализировать процесс обучения модели, построение разделяющей гиперплоскости и выделение опорных векторов.

### Описание данных

Исходные данные включают 7 точек в двумерном пространстве:

- Класс **+1**: (1, 2), (2, 3), (3, 3)
- Класс **-1**: (2, 1), (3, 1), (3, 2), (3, 2.5)

### Параметры модели

Модель обучалась с использованием `SVC` из библиотеки `sklearn` с параметрами:

- `kernel='linear'` — используется линейное ядро (поиск прямой в 2D).
- `C=1000` — высокий коэффициент регуляризации, что требует строгого разделения классов.

После обучения из модели извлекаются параметры гиперплоскости:

- Вектор весов `w`
- Свободный член `b`

Уравнение гиперплоскости имеет вид:
w₁ * x₁ + w₂ * x₂ + b = 0



In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC

# Данные
X = np.array([[1, 2], [2, 3], [3, 3], [2, 1], [3, 1], [3, 2], [3, 2.5]])
y = np.array([1, 1, 1, -1, -1, -1, -1])

# Модель SVM с линейным ядром
svm_model = SVC(kernel='linear', C=1000)
svm_model.fit(X, y)

# Параметры гиперплоскости
w = svm_model.coef_[0]
b = svm_model.intercept_[0]

print(f'Гиперплоскость: {w[0]:.2f}*x₁ + {w[1]:.2f}*x₂ + {b:.2f} = 0')

fig, ax = plt.subplots(figsize=(8, 6))

# Отображаем точки двух классов
ax.scatter(X[y==1][:,0], X[y==1][:,1], color='blue', label='+1')
ax.scatter(X[y==-1][:,0], X[y==-1][:,1], color='red', label='-1')

# Строим разделяющую гиперплоскость
x_plot = np.linspace(0.5, 3.5, 100)
y_plot = -(w[0]/w[1])*x_plot - b/w[1]
ax.plot(x_plot, y_plot, 'k-', label='Гиперплоскость')

# Опорные векторы
ax.scatter(svm_model.support_vectors_[:,0], svm_model.support_vectors_[:,1],
           s=150, linewidth=1.5, facecolors='none', edgecolors='k', label='Опорные векторы')

ax.legend()
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_title('SVM Decision Boundary и Опорные Векторы')
plt.grid()
plt.show()




### Результаты

На итоговом графике показаны:

- Точки двух классов: синие — класс `+1`, красные — класс `-1`.
- **Гиперплоскость** — белая линия, разделяющая классы.
- **Опорные векторы** — точки, расположенные ближе всего к границе. Они обведены белыми кружками и играют ключевую роль в построении разделяющей прямой.

Метод SVM стремится не только разделить классы, но и **максимизировать отступ (margin)** между ними, что повышает устойчивость модели.

### Вывод

Метод SVM успешно построил разделяющую гиперплоскость для линейно разделимых данных. Визуализация иллюстрирует выбор границы и подчёркивает значимость опорных векторов при обучении модели.
